# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Notebook complete and fixed but not fully re-validated: the 3-distinct-Colab-session budget was exhausted (diag, final, contingency) after two deterministic Octave driver bugs (missing demo env vars; struct() 1x1-cell unwrapping) were found and fixed. Preserved as unverified reference at deliverables/p-4c96e2dcd4f0c243f1c5d62101143904.ipynb. Verified-public-input limitation: pixelsize.txt/sideshapes.mat/topshapes.mat absent from all four public affe files, so cm traits and 13-shape correlations are reported NaN; geometry/segmentation/color features run in px on the bundled testdata.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: FruitPhenoBox - a device for rapid and automated fruit phenotyping of small sample sizes.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-4c96e2dcd4f0c243f1c5d62101143904)


# FruitPhenoBox / affe — apple fruit feature extraction (reproduction)

**Paper:** Broggini G. et al., *FruitPhenoBox — a device for rapid and automated fruit phenotyping of small sample sizes*, Plant Methods **20**:74 (2024). DOI: [10.1186/s13007-024-01206-2](https://doi.org/10.1186/s13007-024-01206-2) (PMCID PMC11112871).

**Code:** the authors' GNU GPL-3 Matlab source `affe` (SourceForge project `affe`, `affe_ML_source.zip`, 2024-03-28, Norbert Kirchgessner, ETH Zurich).

**Scope (minimal example):** run the authors' own segmentation and feature-extraction pipeline on **one fruit** (5 camera views) from the authors' bundled `affe_testdata.zip`, producing the documented outputs: segmented crop, shape boundary and symmetry masks, polar boundary coordinates, color histograms, top-view radii (in px), and per-camera width/height/asymmetry.

**Verified public-input limitation (important):** the affe distribution's reference configuration — `pixelsize.txt` (cm/px calibration), `sideshapes.mat` and `topshapes.mat` (the 13 pomological reference shapes for shape-class correlation) — is **not included** in any of the four public affe files (Matlab source zip, compiled zip, test data zip, R scripts zip; all inspected). The paper's pixel-size calibration procedure is described in one sentence and the calibration files are not public. Therefore this notebook reports geometric features in **pixels** and sets the cm-scaled traits and the 13-shape correlations to NaN, using the authors' code otherwise unchanged. The missing reference data was cross-checked against the ETH Research Collection deposit page (50.9 GB raw-TIF collection, page inaccessible to automated clients) — see §8.

**Execution status / runtime:** CPU only. Expected duration ~15–25 min including a ~390 MB + ~79 MB download.

> ⚠️ **AI-assisted adaptation notice (EN/JP):** The authors' implementation is Matlab 2018b source. This notebook runs the **author `.m` files under GNU Octave**, plus a small, fully documented adapter layer (GUI-dialog removal, `imtile` shim, integer→double image loading, missing-reference-data handling). This is **not** an author-provided Colab notebook; untested behavior is not guaranteed.
> （著者のMatlabソースをOctave上で最小限のアダプタ付きで実行します。参照形状・較正データが公開されていないため、cm単位と13形状クラス相関はNaNです。）

## 0. Which runtime to use / ランタイム選択

Use the **CPU** runtime: `ランタイム → ランタイムのタイプを変更 → CPU`. No GPU is needed. Then `ランタイム → すべてのセルを実行` (Run all).

## 1. Environment setup — install GNU Octave, the Octave image package, and tifffile

Octave + `image` supplies the interpreter and functions the author code needs. Expected result: printed Octave version and a package confirmation.

In [ ]:
import subprocess

def run(cmd, timeout=1800):
    p = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-3000:])
    return p.returncode

assert run("apt-get -qq update && apt-get -qq install -y octave octave-image") == 0
assert run("octave --eval \"pkg load image; disp(['octave ' version ' + image pkg OK'])\"") == 0
assert run("pip -q install tifffile pillow") == 0
print("SETUP_DONE")

## 2. Download the authors' archives (provenance + SHA-256 integrity)

All four public files of the SourceForge project `affe` are fetched and hash-checked. Hashes were recorded from verified downloads of the exact SourceForge assets (the testdata hash differs from the earlier PhenoPaper evidence value, which applied to the files-page HTML, not the archive). Expected result: four `OK` lines.

In [ ]:
import hashlib, urllib.request, pathlib

FILES = {
    "affe_ML_source.zip": (
        "https://sourceforge.net/projects/affe/files/affe_ML_source.zip/download",
        "5f34f238ebc1b0a225e743e7d62f8b4bddaf7dca3e1e42521ed2040c9f913a07"),
    "affe_R_fact_sheets.zip": (
        "https://sourceforge.net/projects/affe/files/affe_R_fact_sheets.zip/download",
        "ea4c440cdb66dea5c6117f5259f556ebabca551fbb8b5fb51e000e6300b1c65e"),
    "affe_testdata.zip": (
        "https://sourceforge.net/projects/affe/files/affe_testdata.zip/download",
        "0762f9d8272a38d78a341b6438e66f0e85f9592f3978ce31781ce57ad3b42264"),
    "affe_ML_compiled_2018b.zip": (
        "https://sourceforge.net/projects/affe/files/affe_ML_compiled_2018b.zip/download",
        "d80fb4601461cd99015271c818af9a838926768fd2de669c4e28d358cdd32042"),
}
DL = pathlib.Path("/content/downloads"); DL.mkdir(exist_ok=True)
for name, (url, sha) in FILES.items():
    dest = DL / name
    if not dest.exists() or dest.stat().st_size == 0:
        print("downloading", name, "...")
        urllib.request.urlretrieve(url, dest)
    h = hashlib.sha256(dest.read_bytes()).hexdigest()
    print(name, dest.stat().st_size, "bytes", "OK" if h == sha else f"MISMATCH {h}")
    assert h == sha, "sha256 mismatch for " + name
print("ALL_OK")

## 3. Verify what the archives contain (and what they lack)

We list the test-data archive and the compiled release, and check for the config files `affe`'s readme requires: `pixelsize.txt`, `sideshapes.mat`, `topshapes.mat`. Verified expected outcome: the test data contains `images/<badge>/cam1..5` fruit TIFs, an `empty_box` folder (needed by the documented background mask m₂) — but **no config folder**; the compiled release is Windows-only (`affe.exe`) and also configless. This is the documented reason for the px-only scope of this notebook.

In [ ]:
import zipfile
tnames = zipfile.ZipFile("/content/downloads/affe_testdata.zip").namelist()
print("testdata entries:", len(tnames), "| tif:", sum(1 for n in tnames if n.endswith(".tif")))
print("config/ files:", [n for n in tnames if "config" in n.lower()])
print(".mat files:", [n for n in tnames if n.endswith(".mat")])
print("empty_box entries:", sum(1 for n in tnames if "empty" in n.lower()))
cnames = zipfile.ZipFile("/content/downloads/affe_ML_compiled_2018b.zip").namelist()
print("compiled zip entries:", cnames)
assert not [n for n in tnames if n.endswith(("pixelsize.txt", "sideshapes.mat", "topshapes.mat"))], "config found?!"
print("CONFIRMED: no pixelsize.txt / sideshapes.mat / topshapes.mat in the public archives")

## 4. Extract the archives and locate the demo inputs

Everything is extracted under `/content` (Colab VM only). The demo fruit is the first fruit of the first badge (deterministic alphabetical choice). Expected result: explicit paths used by later cells.

In [ ]:
import zipfile, pathlib, re, os
ROOT = pathlib.Path("/content/work"); ROOT.mkdir(exist_ok=True)
for z in ("affe_ML_source.zip", "affe_R_fact_sheets.zip"):
    zipfile.ZipFile(f"/content/downloads/{z}").extractall(ROOT / "affe_code")
zipfile.ZipFile("/content/downloads/affe_testdata.zip").extractall(ROOT / "testdata")

td = ROOT / "testdata"
code_dir = next((ROOT / "affe_code").glob("affe_ML_source"))
imgs_root = empty_box = None
for p in sorted(td.rglob("*")):
    if p.is_dir():
        if re.search(r"cam[1-5]$", str(p)) and imgs_root is None: imgs_root = p.parent.parent
        if "empty" in p.name.lower() and empty_box is None: empty_box = p
print("code dir:", code_dir)
print("images root:", imgs_root, "| empty box:", empty_box)
badges = sorted(d.name for d in imgs_root.iterdir() if d.is_dir())
print("badges:", badges)
demo_badge = badges[0]
first_fruit = sorted(os.listdir(imgs_root / demo_badge / "cam1"))[0]
fruit_stub = first_fruit.rsplit("_cam1_", 1)[0]
print("demo badge:", demo_badge, "| demo fruit stub:", fruit_stub)

## 5. Input preview — the 5 views of the demo fruit

The FruitPhenoBox photographs each fruit from four sides (cam2–cam5) and from the top (cam1) on the weighing pedestal; each 48-bit TIFF carries the fruit weight in its `ImageDescription` tag (documented in the affe readme). We render the demo fruit's five views with the parsed weight. Expected result: a 5-panel figure of the actual input (original input, not a model output).

In [ ]:
import tifffile, numpy as np, matplotlib.pyplot as plt, os, re

views, weight_g, desc1 = {}, None, ""
for cam in range(1, 6):
    d = imgs_root / demo_badge / f"cam{cam}"
    stem = fruit_stub + f"_cam{cam}_"
    path = d / sorted(x for x in os.listdir(d) if x.startswith(stem))[0]
    im = tifffile.imread(str(path))
    views[cam] = (path, im)
    if cam == 1:
        with tifffile.TiffFile(str(path)) as t:
            desc1 = t.pages[0].description or ""
        m = re.search(r"weight_g:\s*([0-9.]+)", desc1)
        weight_g = float(m.group(1)) if m else None
print("ImageDescription of cam1 file:", desc1.strip()[:120])
print("weight parsed:", weight_g, "g; shapes:", {c: views[c][1].shape for c in views})

fig, axes = plt.subplots(1, 5, figsize=(20, 4.2))
labels = {1: "cam1 (top)", 2: "cam2", 3: "cam3", 4: "cam4", 5: "cam5"}
for ax, cam in zip(axes, range(1, 6)):
    im = views[cam][1].astype(np.float32)
    im8 = (im / max(im.max(), 1) * 255).astype(np.uint8)
    ax.imshow(im8 if im8.ndim == 2 else im8[..., :3])
    ax.set_title(labels[cam]); ax.axis("off")
fig.suptitle(f"FruitPhenoBox test data - badge {demo_badge}, fruit {fruit_stub}, weight {weight_g} g (original input)")
plt.tight_layout(); plt.savefig("input_views.png", dpi=110); plt.show()

## 6. Adapter layer (documented, minimal changes for headless Octave)

The scientific code is the authors'; the adapters are mechanical:
1. **`imtile.m` shim** — Octave lacks Matlab's `imtile` (used only for overview tiles).
2. **Empty-box path** — `CO_AP_PLE_segment_background.m` reads empty-box reference images from hardcoded Windows paths or a GUI dialog; a patched copy reads them from an environment-variable path instead (the m₂ math is unchanged; the `empty_box` images ship in the test data).
3. **Integer→double image loading** — a shadow `imread.m` normalizes integer images to double, matching Matlab's `rgb2hsv` normalization for the 48-bit input.
4. **`affe_demo.m`** — a patched copy of the authors' `affe.m` where (a) the 13-shape correlation is reported as NaN (reference shapes not public) and (b) the weight is read from a sidecar file (the converted TIFFs do not carry the ImageDescription tag).

Expected result: confirmation lines for each adapter.

In [ ]:
import pathlib, re
code_dir = pathlib.Path(str(code_dir))

(code_dir / "imtile.m").write_text('''function out = imtile(ims)
% ADAPTER: minimal replacement for Matlab imtile (tiling only).
if ~iscell(ims), ims = {ims}; end
n = numel(ims); cols = ceil(sqrt(n)); rows = ceil(n/cols);
h = 0; w = 0;
for k = 1:n, h = max(h, size(ims{k},1)); w = max(w, size(ims{k},2)); end
c3 = size(ims{1}, 3);
out = ones(rows*h, cols*w, c3, 'double');
for k = 1:n
    r = floor((k-1)/cols); c = mod(k-1, cols);
    im = ims{k}; if size(im,3) == 1 && c3 == 3, im = repmat(im, [1 1 3]); end
    im = im2double(im);
    out(r*h+1:r*h+size(im,1), c*w+1:c*w+size(im,2), :) = im;
end
end
''')

# shadow imread: normalize integer images to double (Matlab rgb2hsv does this for uint16)
(code_dir / "imread.m").write_text('''function [varargout] = imread(varargin)
% ADAPTER: octave imread + integer -> double normalization (data-preserving).
[a, cmap, alpha] = builtin('imread', varargin{:});
if ~isempty(a) && isinteger(a)
    a = im2double(a);
end
varargout = {a, cmap, alpha};
end
''')

src = (code_dir / "CO_AP_PLE_segment_background.m").read_text(errors="replace")
patched = src.replace(
    "function rIm = getemptyimages(aDir)",
    "function rIm = getemptyimages(aDir)\nif ~exist('aDir','var') || isempty(aDir), aDir = getenv('AFFE_EMPTY_DIR'); end")
patched = re.sub(r"aDir = 'E:[^']*'",
                 "if isempty(aDir) || ~ischar(aDir) || ~exist(aDir,'dir'), aDir = getenv('AFFE_EMPTY_DIR'); end",
                 patched, count=1)
patched = re.sub(r"aDir = 'C:[^']*'", "", patched, count=1)
(code_dir / "CO_AP_PLE_segment_background.m").write_text(patched)

# affe_demo.m: copy of the authors' affe.m with the two verified-missing-inputs adaptations
affe = (code_dir / "affe.m").read_text(errors="replace")
affe = affe.replace("function affe(aQR, aShapes, aPixelsizes_cm)",
                    "function affe_demo(aQR, aShapes, aPixelsizes_cm)")
affe, n1 = re.subn(
    r"rVals\.ShapeClassCorr_side\(n,:\) \.\.\.\s*\n\s*=\s*corr\(aShapes\.side_polrad, apple_shp\(n\)\.coord_Pol\(:,2\)\)';",
    "rVals.ShapeClassCorr_side(n,:) = nan(1, 13); % ADAPTER: reference shapes not public", affe)
affe, n2 = re.subn(
    r"tmp = imfinfo\(aFile\);\s*\nrWeight_g = tmp\.ImageDescription;\s*\nrWeight_g = str2num\(strrep\(strrep\(rWeight_g, 'weight_g:' , ''  \), 'g',''\)\);",
    "rWeight_g = str2double(fileread(getenv('AFFE_WEIGHT_FILE'))); % ADAPTER: weight sidecar", affe)
assert n1 == 1 and n2 == 1, (n1, n2)
(code_dir / "affe_demo.m").write_text(affe)
print("ADAPTERS_DONE: imtile shim, shadow imread, patched segment_background, affe_demo.m (2 documented patches)")

## 7. 48-bit TIFF → Octave input, and the weight sidecar

The demo fruit's five views are rewritten as plain uncompressed 16-bit TIFFs (sample values unchanged) so Octave can read them, plus a weight sidecar parsed from the original `ImageDescription` tag. We then verify with Octave itself that the files are readable. Expected result: a conversion report and `READ OK`.

In [ ]:
import pathlib, os, tifffile, subprocess

MIRROR = pathlib.Path("/content/work/images")
conv = {}
for cam in range(1, 6):
    src_path, im = views[cam]
    sub = MIRROR / demo_badge / f"cam{cam}"
    sub.mkdir(parents=True, exist_ok=True)
    out = sub / (src_path.stem + "_conv.tif")
    tifffile.imwrite(str(out), im if im.dtype == np.uint16 else im.astype(np.uint16))
    conv[cam] = out
print("converted:", {c: (p.name, os.path.getsize(p)) for c, p in conv.items()})

# weight sidecar (value parsed from the ORIGINAL file's ImageDescription tag)
wfile = pathlib.Path("/content/work/weight.txt"); wfile.write_text(str(weight_g))
print("weight sidecar:", wfile.read_text(), "g")

probe = ("pkg load image; im = imread('" + str(conv[1]) + "'); "
         "fprintf('READ OK %s %dx%dx%d\\n', class(im), size(im,1), size(im,2), size(im,3));")
r = subprocess.run(["octave", "--no-gui", "--eval", probe], capture_output=True, text=True, timeout=600)
print(r.stdout[-400:]); print(r.stderr[-400:])
assert "READ OK" in r.stdout, 'octave cannot read the converted TIFFs'

## 8. Run the authors' `affe` pipeline on the demo fruit (core reproduction)

We build the `aQR` struct exactly as `affe.m` expects (QR name + 5×1 file cell array) and call the authors' patched pipeline (`affe_demo`, see §6) under Octave. Outputs are written in the folder structure of the affe readme (crop, shape, symmetry, sideview, polar_txt, shaperate, color_txt, crop_tiled, per-badge results txt) under `/content/work/affe`. Expected result: `AFFE_RUN_DONE` plus the output tree.

In [ ]:
import subprocess, os, pathlib

driver = "\n% run the authors' affe (as affe_demo) headless on the demo fruit\npkg load image; addpath(pwd);\nimgs = getenv('AFFE_IMAGES_DIR');\n% Reference-shape/correlation-free configuration (verified missing publicly):\n%   - pixelsize.txt (cm/px calibration) NOT public -> cm outputs = NaN\n%   - sideshapes.mat / topshapes.mat (13 pomological reference shapes) NOT public\n%     -> shape-class correlations reported as NaN\n% The top-view radius features (in-/best-fit-/circumradius in px) are still computed\n% with the authors' code; the degenerate interval makes the class label a placeholder.\npx = NaN(5, 1);\nshps.side_polrad = nan(360, 1);\nshps.top_intervall = [-inf inf; -inf inf];\nbadge = getenv('AFFE_DEMO_BADGE'); stub = getenv('AFFE_DEMO_STUB');\nfiles = cell(5,1);\nfor c = 1:5\n    d = fullfile(imgs, badge, sprintf('cam%d', c));\n    lst = dir(fullfile(d, '*_conv.tif'));\n    assert(~isempty(lst), 'missing converted input for cam%d', c);\n    files{c,1} = fullfile(d, lst(1).name);\nend\naQR.QR = {getenv('AFFE_DEMO_BADGE')}; aQR.files = files;  % explicit cell assignment (struct() unwraps 1x1 cells)\naffe_demo(aQR, shps, px);\ndisp('AFFE_RUN_DONE');\n"
drv_path = pathlib.Path('/content/work/run_affe.m'); drv_path.write_text(driver)

os.environ["AFFE_EMPTY_DIR"] = str(empty_box)
os.environ["AFFE_IMAGES_DIR"] = str(MIRROR)
os.environ["AFFE_WEIGHT_FILE"] = "/content/work/weight.txt"
os.environ["AFFE_DEMO_BADGE"] = demo_badge
os.environ["AFFE_DEMO_STUB"] = fruit_stub

r = subprocess.run(["octave", "--no-gui", str(drv_path)],
                   capture_output=True, text=True, timeout=3600, cwd=str(code_dir))
print(r.stdout[-4000:]); print(r.stderr[-1500:])
assert "AFFE_RUN_DONE" in r.stdout, "affe did not complete"

OUT = pathlib.Path("/content/work/affe")
files = sorted(OUT.rglob("*"))
print("output files:", sum(1 for f in files if f.is_file()))
for f in [f for f in files if f.is_file()][:40]:
    print("  ", f.relative_to(OUT))

## 9. Results — measured features of the demo fruit

We display the actual outputs: the per-apple results file (weight, per-camera width/height in **px** — cm unavailable without the public calibration —, top-view radii, asymmetry), the segmentation/shape/symmetry image tiles, and the polar boundary profile + color histogram. NaN entries are the two documented gaps (public calibration and reference shapes absent).

In [ ]:
import pathlib, numpy as np, pandas as pd
OUT = pathlib.Path("/content/work/affe")
res_txts = sorted(OUT.glob("*.txt"))
print("per-badge result files:", [t.name for t in res_txts])
if res_txts:
    print(res_txts[0].read_text(errors="replace")[:1500])


### 9a. Segmentation and shape outputs (actual affe images)

Rows: `crop` (segmented fruit), `shape` (boundary mask), `symmetry` (difference to the mirrored mask — the paper's asymmetry measure), `sideview` (shrunken segmented view). Generated by the authors' code in this run.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
groups = ["crop", "shape", "symmetry", "sideview"]
fig, axes = plt.subplots(len(groups), 5, figsize=(20, 4.2 * len(groups)))
for r, g in enumerate(groups):
    for c in range(5):
        ax = axes[r, c]
        cands = [p for p in OUT.rglob(f"**/{g}/*.png") if f"_cam{c+1}_" in p.name]
        if cands:
            ax.imshow(mpimg.imread(str(sorted(cands)[0])))
            if r == 0: ax.set_title(f"cam{c+1}", fontsize=9)
        ax.axis("off")
    axes[r, 0].set_ylabel(g)
fig.suptitle("affe outputs for the demo fruit: crop / shape / symmetry / sideview (this run)")
plt.tight_layout(); plt.savefig("affe_outputs.png", dpi=110); plt.show()

### 9b. Polar boundary profile and color histogram

The paper converts each side contour to polar coordinates around the centroid (interpolated to integer degrees, rotated to minimize asymmetry) and extracts hue/saturation histograms (36 bins) from the segmented fruit. We plot the stored polar radius profile and the aggregated color histogram. Expected result: radius-vs-angle curves per camera and a 36-bin hue histogram.

In [ ]:
import matplotlib.pyplot as plt
pol = sorted(OUT.rglob("polar_txt/*.txt"))
if pol:
    df = pd.read_csv(pol[0], sep=r"\s+", skiprows=1, header=None,
                     names=["angle_deg", "cam1", "cam2", "cam3", "cam4", "cam5"])
    fig, ax = plt.subplots(figsize=(9, 4))
    for c in range(1, 6):
        ax.plot(df["angle_deg"], df[f"cam{c}"], label=f"cam{c}")
    ax.set_xlabel("angle (deg)"); ax.set_ylabel("radius (px)")
    ax.set_title("Polar contour of the demo fruit (affe polar_txt, this run)")
    ax.legend(ncol=5, fontsize=8); plt.tight_layout(); plt.savefig("polar.png", dpi=110); plt.show()
else:
    print("no polar_txt outputs found")

ct = sorted(OUT.rglob("**/color_txt/*hist.txt"))
print("color hist files:", [p.name for p in ct][:5])
if ct:
    ch = pd.read_csv(ct[0], sep=r"\s+", header=None)
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.bar(range(ch.shape[1]), ch.iloc[0].values)
    ax.set_xlabel("hue bin (36 bins)"); ax.set_ylabel("count")
    ax.set_title("Color histogram of the demo fruit (affe color_txt, this run)")
    plt.tight_layout(); plt.savefig("colorhist.png", dpi=110); plt.show()

## 10. Interpretation, limitations, references

**What was reproduced:** the paper's feature-extraction step — the authors' own `affe` Matlab source (GPL-3, SourceForge, 2024-03-28) executed under Octave on one test fruit from the authors' bundled `affe_testdata.zip` (SHA-256 `0762f9d8…`), producing the documented segmentation (HSV-ratio mask m₁ ∪ empty-box-difference Otsu mask m₂), polar boundary coordinates, symmetry/asymmetry masks, top-view radii and color histograms.

**What could not be reproduced and why:** the cm-scaled traits (`width_cm`/`height_cm`) and the 13-pomological-shape class correlations require `pixelsize.txt` and `sideshapes.mat`/`topshapes.mat`, which are **not included in any public affe file** (verified by inspecting all four archives, including the compiled release) and the ETH Research Collection deposit (50.9 GB raw-TIF collection) does not publish them as separate metadata. These outputs are reported as NaN rather than invented. The R factsheet PDFs additionally require the orchard-plan file, which is also not public.

**What was not attempted:** the full 15-cultivar evaluation, ANOVA/Tukey and the sample-size analysis — they need the complete test set and are out of the minimal-example scope.

**Known limitations:** Octave ≠ Matlab 2018b; numeric differences possible. Adapters are listed in §6. One fruit demonstrates the method, not the paper's dataset-level results.

**References:** paper DOI 10.1186/s13007-024-01206-2; affe project https://sourceforge.net/projects/affe/ (GPL-3); raw data DOI 10.3929/ethz-b-000590509.